# Train-Ready Grammar Data EDA

This notebook focuses on the final cleaned dataset used for BabyLLaMA fine-tuning.

It is designed for files such as `train_ready_grammar_data.jsonl`, and it uses the same official `count_word` method from `Language_acquisition.ipynb` to count words in `good` and `bad`.

In [ ]:
import json
from pathlib import Path

import nltk
import pandas as pd
from nltk.tokenize import word_tokenize

In [ ]:
for resource in ("punkt", "punkt_tab"):
    try:
        nltk.data.find(f"tokenizers/{resource}")
    except LookupError:
        nltk.download(resource)

In [ ]:
TRAIN_READY_PATH = Path("train_ready_grammar_data.jsonl")

In [ ]:
def count_word(text):
    tokens = word_tokenize(text)
    return len(tokens)

In [ ]:
def load_jsonl(path):
    records = []
    with path.open(encoding="utf-8") as f:
        for line in f:
            line = line.strip()
            if not line:
                continue
            records.append(json.loads(line))
    return records

In [ ]:
records = load_jsonl(TRAIN_READY_PATH)
df = pd.DataFrame(records)
print(f"Loaded {len(df)} train-ready records from {TRAIN_READY_PATH}")
df.head()

## Required Columns

In [ ]:
required_columns = [
    "id",
    "phenomenon",
    "topic",
    "good",
    "bad",
    "edit_type",
    "prompt_family",
    "parent_llm",
]

missing_columns = [col for col in required_columns if col not in df.columns]
missing_columns

## Official Word Counts for `good` and `bad`

In [ ]:
df["good_word_count"] = df["good"].fillna("").map(count_word)
df["bad_word_count"] = df["bad"].fillna("").map(count_word)
df["pair_word_count"] = df["good_word_count"] + df["bad_word_count"]
df[["good", "bad", "good_word_count", "bad_word_count", "pair_word_count"]].head()

In [ ]:
word_count_summary = {
    "num_records": int(len(df)),
    "total_good_words": int(df["good_word_count"].sum()),
    "total_bad_words": int(df["bad_word_count"].sum()),
    "total_pair_words": int(df["pair_word_count"].sum()),
    "avg_good_words": float(df["good_word_count"].mean()) if len(df) else 0.0,
    "avg_bad_words": float(df["bad_word_count"].mean()) if len(df) else 0.0,
    "avg_pair_words": float(df["pair_word_count"].mean()) if len(df) else 0.0,
    "min_good_words": int(df["good_word_count"].min()) if len(df) else 0,
    "max_good_words": int(df["good_word_count"].max()) if len(df) else 0,
    "min_bad_words": int(df["bad_word_count"].min()) if len(df) else 0,
    "max_bad_words": int(df["bad_word_count"].max()) if len(df) else 0,
}
word_count_summary

## Dataset Overview

In [ ]:
overview = {
    "num_records": int(len(df)),
    "num_unique_ids": int(df["id"].nunique(dropna=True)),
    "num_unique_good": int(df["good"].nunique(dropna=True)),
    "num_unique_bad": int(df["bad"].nunique(dropna=True)),
    "num_unique_phenomena": int(df["phenomenon"].nunique(dropna=True)),
    "num_unique_topics": int(df["topic"].nunique(dropna=True)),
    "num_unique_edit_types": int(df["edit_type"].nunique(dropna=True)),
    "num_unique_prompt_families": int(df["prompt_family"].nunique(dropna=True)),
    "num_unique_parent_llms": int(df["parent_llm"].nunique(dropna=True)),
}
overview

In [ ]:
df.isna().sum().sort_values(ascending=False)

## Counts by Phenomenon

In [ ]:
df["phenomenon"].value_counts().rename_axis("phenomenon").reset_index(name="count")

## Counts by Topic

In [ ]:
df["topic"].value_counts().rename_axis("topic").reset_index(name="count")

## Counts by Parent LLM

In [ ]:
df["parent_llm"].value_counts().rename_axis("parent_llm").reset_index(name="count")

## Counts by Prompt Family

In [ ]:
df["prompt_family"].value_counts().rename_axis("prompt_family").reset_index(name="count")

## Counts by Edit Type

In [ ]:
df["edit_type"].value_counts().rename_axis("edit_type").reset_index(name="count")

## Cross Tabs

In [ ]:
pd.crosstab(df["phenomenon"], df["parent_llm"])

In [ ]:
pd.crosstab(df["phenomenon"], df["prompt_family"])

In [ ]:
pd.crosstab(df["topic"], df["parent_llm"])

## Length Analysis

In [ ]:
df[["good_word_count", "bad_word_count", "pair_word_count"]].describe()

In [ ]:
df["word_count_gap"] = (df["good_word_count"] - df["bad_word_count"]).abs()
df["word_count_gap"].describe()

In [ ]:
df.sort_values("pair_word_count", ascending=False)[
    ["id", "phenomenon", "topic", "good", "bad", "good_word_count", "bad_word_count", "pair_word_count"]
].head(20)

## Quality Checks on Final Training Data

In [ ]:
quality_checks = {
    "good_equals_bad": int((df["good"] == df["bad"]).sum()),
    "duplicate_ids": int(df.duplicated(subset=["id"]).sum()),
    "duplicate_training_rows": int(df.duplicated(subset=["phenomenon", "topic", "good", "bad", "edit_type"]).sum()),
    "empty_good": int((df["good"].fillna("").str.strip() == "").sum()),
    "empty_bad": int((df["bad"].fillna("").str.strip() == "").sum()),
}
quality_checks

In [ ]:
df[df.duplicated(subset=["phenomenon", "topic", "good", "bad", "edit_type"], keep=False)].sort_values(
    ["phenomenon", "topic", "edit_type"]
)[["id", "phenomenon", "topic", "good", "bad", "edit_type", "prompt_family", "parent_llm"]].head(30)

## Samples

In [ ]:
df.sample(min(10, len(df)), random_state=42) if len(df) else df

In [ ]:
for phenomenon in sorted(df["phenomenon"].dropna().unique()[:5]):
    print(f"\n=== {phenomenon} ===")
    display(df[df["phenomenon"] == phenomenon][["id", "topic", "good", "bad", "edit_type"]].head(3))

## Optional Export

In [ ]:
EXPORT_CSV = False

if EXPORT_CSV:
    df.to_csv("train_ready_grammar_data_eda_export.csv", index=False)
    print("Saved train_ready_grammar_data_eda_export.csv")